# 🧬 Pipeline de Vetorização Espectral SWeeP 600D (Dataset Fujita)

Este notebook implementa o pipeline de alta performance para:
1. **Carregamento OOM-Safe** da matriz transcriptômica Fujita (`matrizFiltradaeNormalizadaF (1).h5ad`) e rótulos (`cell_types_binarioF.txt`);
2. **Filtragem das Classes Canônicas** (classes 1 a 7, descartando a classe 0 e classes residuais), reindexadas para 0 a 6 para compatibilidade nativa com Keras;
3. **Binarização Ultrarrápida** da expressão gênica ($>0 \to 1$, $0 \to 0$) diretamente em formato esparso (`CSR`), consumindo mínimo de memória RAM;
4. **Geração / Congelamento da Ortobase Fixa SWeeP** ($36.601 \times 600$) via decomposição ortonormal determinística (`seed=42`);
5. **Projeção Dimensional SWeeP 600D** ($W_{swp} = X_{bin} \times B$) em segundos;
6. **Exportação dos Artefatos Prontos** em formato NumPy binário (`.npy`) e metadados (`.json`) para alimentação direta no classificador MLP Keras.


## ⚙️ Configuração do Ambiente e Definição de Parâmetros


In [ ]:
import json
import time
from pathlib import Path

import anndata as ad
import numpy as np
import scipy.sparse as sp

# Parâmetros Fundamentais do Experimento
SEED = 42
DIM_PROJECAO = 600
CLASSES_VALIDAS = list(range(1, 8))  # Classes 1 a 7

# Caminhos de Entrada e Saída
PATH_DADOS = Path("dados")
PATH_H5AD = PATH_DADOS / "matrizFiltradaeNormalizadaF (1).h5ad"
PATH_LABELS = PATH_DADOS / "cell_types_binarioF.txt"

PATH_X_OUT = PATH_DADOS / "X_fujita_600d.npy"
PATH_Y_OUT = PATH_DADOS / "y_fujita.npy"
PATH_ORTHBASE_OUT = PATH_DADOS / "orthbase_fujita_600d.npy"
PATH_METADATA_OUT = PATH_DADOS / "metadata_classes.json"

print("✓ Configurações carregadas com sucesso!")
print(f"  • H5AD de entrada  : {PATH_H5AD}")
print(f"  • Rótulos          : {PATH_LABELS}")
print(f"  • Dimensão SWeeP   : {DIM_PROJECAO}")
print(f"  • Semente Fixa     : {SEED}")

✓ Configurações carregadas com sucesso!
  • H5AD de entrada  : dados/matrizFiltradaeNormalizadaF (1).h5ad
  • Rótulos          : dados/cell_types_binarioF.txt
  • Dimensão SWeeP   : 600
  • Semente Fixa     : 42


## 🏷️ Etapa 1: Carregamento dos Rótulos e Filtragem das Classes 1 a 7

Lemos o arquivo `cell_types_binarioF.txt` e construímos a máscara booleana para reter exclusivamente as classes celulares 1 a 7, descartando a classe 0 (não-anotadas) e classes acima de 7. Reindexamos as classes para o intervalo **0 a 6** (formato canônico zero-indexed de funções de perda como `sparse_categorical_crossentropy` do Keras).


In [2]:
t_inicio_etapa = time.time()

with open(PATH_LABELS, "r") as f:
    rotulos_originais = np.array(
        [int(l.strip()) for l in f if l.strip()], dtype=np.int64
    )

total_celulas_bruto = len(rotulos_originais)
print(f"Total de células anotadas no arquivo: {total_celulas_bruto:,}")

# Máscara de seleção para classes canônicas 1 a 7
mascara_classes = np.isin(rotulos_originais, CLASSES_VALIDAS)
total_filtradas = int(mascara_classes.sum())

# Rótulos ajustados para zero-index (0 a 6)
y_filtrado = rotulos_originais[mascara_classes] - 1

print(f"✓ Filtragem concluída em {time.time() - t_inicio_etapa:.2f}s")
print(
    f"  • Células mantidas (classes 1-7) : {total_filtradas:,} ({total_filtradas / total_celulas_bruto * 100:.2f}%)"
)
print(
    f"  • Células descartadas (classe 0/+) : {total_celulas_bruto - total_filtradas:,}"
)
print("  • Distribuição por classe ajustada (0 a 6):")
for cls_idx in range(7):
    qtd = int((y_filtrado == cls_idx).sum())
    print(f"    - Classe Keras {cls_idx} (Original {cls_idx + 1}): {qtd:,} células")

Total de células anotadas no arquivo: 40,913
✓ Filtragem concluída em 0.01s
  • Células mantidas (classes 1-7) : 26,763 (65.41%)
  • Células descartadas (classe 0/+) : 14,150
  • Distribuição por classe ajustada (0 a 6):
    - Classe Keras 0 (Original 1): 4,366 células
    - Classe Keras 1 (Original 2): 188 células
    - Classe Keras 2 (Original 3): 9,073 células
    - Classe Keras 3 (Original 4): 4,567 células
    - Classe Keras 4 (Original 5): 1,734 células
    - Classe Keras 5 (Original 6): 5,762 células
    - Classe Keras 6 (Original 7): 1,073 células


## 📦 Etapa 2: Carregamento do Dataset Transcriptômico Fujita (`.h5ad`)

Carregamos o dataset AnnData e aplicamos a máscara booleana celular para carregar apenas as 26.763 células selecionadas na memória de trabalho.


In [3]:
t_inicio_etapa = time.time()
print(f"Carregando matriz AnnData: {PATH_H5AD}...")

adata = ad.read_h5ad(PATH_H5AD)
print(f"✓ Dataset carregado em {time.time() - t_inicio_etapa:.2f}s!")
print(f"  • Dimensões originais: {adata.shape[0]:,} células × {adata.shape[1]:,} genes")
print(f"  • Tipo de representação: {type(adata.X)}")

# Aplicação da máscara de classes diretamente na matriz esparsa
X_filtrado = adata.X[mascara_classes]
nomes_genes = list(adata.var_names)
barcodes_filtrados = list(adata.obs_names[mascara_classes])

print(
    f"  • Dimensões filtradas: {X_filtrado.shape[0]:,} células × {X_filtrado.shape[1]:,} genes"
)

# Liberamos o objeto AnnData completo para otimização máxima de RAM
del adata

Carregando matriz AnnData: dados/matrizFiltradaeNormalizadaF (1).h5ad...


✓ Dataset carregado em 2.24s!
  • Dimensões originais: 40,913 células × 36,601 genes
  • Tipo de representação: <class 'scipy.sparse._csc.csc_matrix'>


  • Dimensões filtradas: 26,763 células × 36,601 genes


## ⚡ Etapa 3: Binarização In-Place OOM-Safe da Expressão Gênica

Conforme requisito da pesquisa, aplicamos a binarização:
- Expressão $> 0 \implies 1$
- Expressão $= 0 \implies 0$

> **Otimização OOM-Safe:** Em uma matriz esparsa (`CSR` / `CSC`), apenas os valores não-nulos são mantidos no array `.data`. Em vez de transformar os dados em matriz densa (o que exigiria mais de 6 GB de RAM), alteramos diretamente os valores do vetor esparso `.data` para $1.0$, executando a operação em fração de segundo e sem consumo extra de memória.


In [4]:
t_inicio_etapa = time.time()

# Asseguramos formato CSR (Compressed Sparse Row) para máxima velocidade de produto de matrizes
if sp.issparse(X_filtrado):
    X_bin = X_filtrado.tocsr()
    # Binarização in-place no vetor esparso .data
    X_bin.data = np.ones_like(X_bin.data, dtype=np.float32)
else:
    X_bin = sp.csr_matrix((X_filtrado > 0).astype(np.float32))

print(f"✓ Binarização concluída em {time.time() - t_inicio_etapa:.2f}s!")
print(f"  • Total de entradas ativas (1s): {X_bin.nnz:,}")
print(
    f"  • Densidade da matriz esparsa  : {(X_bin.nnz / (X_bin.shape[0] * X_bin.shape[1])) * 100:.2f}%"
)
print(f"  • Valores únicos em .data      : {np.unique(X_bin.data)}")

✓ Binarização concluída em 3.13s!
  • Total de entradas ativas (1s): 104,708,300
  • Densidade da matriz esparsa  : 10.69%


  • Valores únicos em .data      : [1.]


## 📐 Etapa 4: Geração e Congelamento da Ortobase Fixa SWeeP

O algoritmo **SWeeP** (*Spaced Words Projection*) mapeia os $36.601$ genes para um subespaço de **600 dimensões** através de uma base ortonormal $B \in \mathbb{R}^{36.601 \times 600}$:
$$B^T B = I_{600}$$

Geramos a matriz ortonormal via decomposição QR a partir de uma distribuição normal controlada pela semente determinística `seed = 42`. A base gerada é persistida em `dados/orthobase_fujita_600d.npy` para assegurar **congelamento determinístico** e reprodutibilidade longitudinal.


In [5]:
t_inicio_etapa = time.time()
n_genes = X_bin.shape[1]

if PATH_ORTHBASE_OUT.exists():
    print(
        f"[Auditoria] Carregando ortobase fixa congelada existente em: {PATH_ORTHBASE_OUT}..."
    )
    orthbase = np.load(PATH_ORTHBASE_OUT)
    assert orthbase.shape == (n_genes, DIM_PROJECAO), (
        f"Incompatibilidade dimensional: esperava ({n_genes}, {DIM_PROJECAO}), encontrou {orthbase.shape}"
    )
else:
    print(
        f"Gerando nova ortobase canônica ({n_genes:,} genes × {DIM_PROJECAO} dimensões, seed={SEED})..."
    )
    rng = np.random.default_rng(SEED)
    H = rng.standard_normal((n_genes, DIM_PROJECAO), dtype=np.float32)
    orthbase, _ = np.linalg.qr(H)

    # Persistência da base congelada para reutilização obrigatória
    np.save(PATH_ORTHBASE_OUT, orthbase.astype(np.float32))
    print(f"✓ Ortobase congelada e salva em: {PATH_ORTHBASE_OUT}")

# Verificação de ortonormalidade (B.T @ B ≈ I_600)
I_teste = orthbase.T @ orthbase
erro_ortonormalidade = np.max(np.abs(I_teste - np.eye(DIM_PROJECAO, dtype=np.float32)))
print(f"✓ Base validada em {time.time() - t_inicio_etapa:.2f}s!")
print(f"  • Shape da Ortobase          : {orthbase.shape}")
print(f"  • Erro máx de ortonormalidade: {erro_ortonormalidade:.2e} (perfeita)")

Gerando nova ortobase canônica (36,601 genes × 600 dimensões, seed=42)...


✓ Ortobase congelada e salva em: dados/orthbase_fujita_600d.npy
✓ Base validada em 2.76s!
  • Shape da Ortobase          : (36601, 600)
  • Erro máx de ortonormalidade: 4.77e-07 (perfeita)


## 🚀 Etapa 5: Projeção Espectral SWeeP 600D

Projetamos a matriz binarizada $X_{bin}$ sobre o subespaço ortogonal da base:
$$W_{swp} = X_{bin} \times B \in \mathbb{R}^{26.763 \times 600}$$

Utilizando rotinas otimizadas de BLAS / SciPy para multiplicação de matriz esparsa por densa, a projeção é calculada em cerca de 20 segundos.


In [6]:
t_inicio_etapa = time.time()
print("Iniciando projeção espectral SWeeP (X_bin @ orthbase)...")

X_projetado = X_bin @ orthbase
X_projetado = np.asarray(X_projetado, dtype=np.float32)

print(f"✓ Projeção concluída em {time.time() - t_inicio_etapa:.2f}s!")
print(f"  • Dimensões da matriz projetada : {X_projetado.shape}")
print(f"  • Quantidade de NaNs            : {np.isnan(X_projetado).sum()}")
print(f"  • Quantidade de Infs            : {np.isinf(X_projetado).sum()}")
print(f"  • Média geral dos valores       : {X_projetado.mean():.4f}")
print(f"  • Desvio padrão geral           : {X_projetado.std():.4f}")

Iniciando projeção espectral SWeeP (X_bin @ orthbase)...


✓ Projeção concluída em 21.50s!
  • Dimensões da matriz projetada : (26763, 600)
  • Quantidade de NaNs            : 0
  • Quantidade de Infs            : 0
  • Média geral dos valores       : -0.0018
  • Desvio padrão geral           : 0.3378


## 💾 Etapa 6: Persistência dos Dados de Entrada para a Rede MLP

Salvamos os tensores finais em formato binário NumPy (`.npy`) e um arquivo de metadados descritivo (`.json`). O formato binário permite carregamento em milissegundos sem sobrecarga de I/O em etapas de treinamento e inferência no Keras.


In [7]:
t_inicio_etapa = time.time()

# Salva matriz de features e array de rótulos
np.save(PATH_X_OUT, X_projetado)
np.save(PATH_Y_OUT, y_filtrado)

# Salva metadados do dataset para rastreabilidade científica
metadados = {
    "nome_dataset": "Fujita Filtrado e Normalizado (Binarizado)",
    "algoritmo": "rSWeeP (Spaced Words Projection)",
    "dimensao_latente": DIM_PROJECAO,
    "semente_ortobase": SEED,
    "total_amostras_originais": total_celulas_bruto,
    "total_amostras_mlp": int(X_projetado.shape[0]),
    "total_genes": int(X_bin.shape[1]),
    "mapeamento_classes": {
        f"Classe_Keras_{i}": {
            "label_original": i + 1,
            "label_keras": i,
            "total_amostras": int((y_filtrado == i).sum()),
        }
        for i in range(7)
    },
    "arquivos_gerados": {
        "features_x": str(PATH_X_OUT),
        "labels_y": str(PATH_Y_OUT),
        "ortobase": str(PATH_ORTHBASE_OUT),
    },
}

with open(PATH_METADATA_OUT, "w", encoding="utf-8") as f:
    json.dump(metadados, f, indent=2, ensure_ascii=False)

tam_x_mb = PATH_X_OUT.stat().st_size / (1024 * 1024)
tam_y_kb = PATH_Y_OUT.stat().st_size / 1024
tam_base_mb = PATH_ORTHBASE_OUT.stat().st_size / (1024 * 1024)

print(f"✓ Artefatos persistidos em {time.time() - t_inicio_etapa:.2f}s!")
print(f"  • Features X  : {PATH_X_OUT} ({tam_x_mb:.2f} MB, shape: {X_projetado.shape})")
print(f"  • Labels y    : {PATH_Y_OUT} ({tam_y_kb:.2f} KB, shape: {y_filtrado.shape})")
print(
    f"  • Ortobase    : {PATH_ORTHBASE_OUT} ({tam_base_mb:.2f} MB, shape: {orthbase.shape})"
)
print(f"  • Metadados   : {PATH_METADATA_OUT}")

✓ Artefatos persistidos em 0.01s!
  • Features X  : dados/X_fujita_600d.npy (61.26 MB, shape: (26763, 600))
  • Labels y    : dados/y_fujita.npy (209.21 KB, shape: (26763,))
  • Ortobase    : dados/orthbase_fujita_600d.npy (83.77 MB, shape: (36601, 600))
  • Metadados   : dados/metadata_classes.json


## 🔍 Etapa 7: Verificação e Auditoria de Prontidão para o Keras

Demonstramos a carga instantânea dos arquivos e inspecionamos os dados conforme esperados pelo Keras para treino ou inferência de classificação celular.


In [8]:
# Carga instantânea via NumPy
X_verif = np.load(PATH_X_OUT)
y_verif = np.load(PATH_Y_OUT)

print("=" * 60)
print("          AUDITORIA DE PRONTIDÃO PARA A REDE MLP")
print("=" * 60)
print(f"• X shape        : {X_verif.shape} (esperado: (26763, 600))")
print(f"• X dtype        : {X_verif.dtype} (esperado: float32)")
print(f"• y shape        : {y_verif.shape} (esperado: (26763,))")
print(f"• y dtype        : {y_verif.dtype} (esperado: int64)")
print(f"• y min / max    : {y_verif.min()} / {y_verif.max()} (esperado: 0 a 6)")
print(f"• Classes únicas : {np.unique(y_verif)}")
print(f"• Ausência de NaN: {not np.isnan(X_verif).any()}")
print("=" * 60)
print("🎯 DADOS 100% PRONTOS PARA CLASSIFICAÇÃO NA REDE MLP COM KERAS!")

          AUDITORIA DE PRONTIDÃO PARA A REDE MLP
• X shape        : (26763, 600) (esperado: (26763, 600))
• X dtype        : float32 (esperado: float32)
• y shape        : (26763,) (esperado: (26763,))
• y dtype        : int64 (esperado: int64)
• y min / max    : 0 / 6 (esperado: 0 a 6)
• Classes únicas : [0 1 2 3 4 5 6]
• Ausência de NaN: True
🎯 DADOS 100% PRONTOS PARA CLASSIFICAÇÃO NA REDE MLP COM KERAS!
